In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings

from sklearn import pipeline

warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('./processed_train2.csv')
data.head()

## HW13 — линейный SVM и логистическая регрессия

Разделите данные на train/test; полный CSV не пересчитывался.

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop('HasDetections', axis=1)
y = data['HasDetections']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=33)

Создайте pipe_lr и pipe_svm: MinMaxScaler, затем LogisticRegression или LinearSVC. Обучение выполняется на train.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import MinMaxScaler


### Your code is here
pipe_lr = Pipeline([('MinMaxScaler',MinMaxScaler()),
                        ('LR', LogisticRegression())])
pipe_svm = Pipeline([('MinMaxScaler',MinMaxScaler()),
                              ('SVM', LinearSVC())])


In [ ]:
### Обучаем модели на трейне
### Your code is here

pipe_lr.fit(X_train, y_train)

pipe_svm.fit(X_train, y_train)

print(pipe_lr.score(X_test, y_test))
print(pipe_svm.score(X_test, y_test))

### ROC и преобразование оценок SVM

Сравните ROC-AUC моделей на test. Для SVM по условию требуется кусочно-пропорциональное преобразование decision_function: отрицательные оценки в [0,0.5), положительные в (0.5,1], ноль в 0.5. Минимальная отрицательная оценка отображается в 0, максимальная положительная в 1. Например, для [-10,-5,0,2,10,15] результаты [0,0.25,0.5,0.5667,0.8333,1]. Это учебное преобразование, а не доказательство калибровки вероятностей.

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import RocCurveDisplay

In [ ]:
### С изображением рок-кривой для ЛогРега не будет проблем:
### Your code is here
from sklearn.metrics import auc

fpr, tpr, thresholds = roc_curve(y_test, pipe_lr.predict_proba(X_test)[:,1])

RocCurveDisplay(fpr=fpr,tpr=tpr).plot()

auc(fpr,tpr)


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay


### А для того, чтобы построить и изобразить рок-кривую для SVM, придется
### в начале преобразовать выходы pipe_svm.decision_function

### Your code is here
# pipe_svm.decision_function(X_test.head(2))

def sigmoid(output):
    # Имя функции сохранено; преобразование пропорциональное по условию.
    output = np.asarray(output)
    result = np.full_like(output, 0.5, dtype=float)
    negative = output < 0
    positive = output > 0
    if negative.any():
        result[negative] = 0.5 * (output[negative] - output[negative].min()) / abs(output[negative].min())
    if positive.any():
        result[positive] = 0.5 + 0.5 * output[positive] / output[positive].max()
    return result

pred_prob = sigmoid(pipe_svm.decision_function(X_test))

# PrecisionRecallDisplay.from_estimator(pipe_svm, X_test, y_test)
fpr, tpr, thresholds = roc_curve(y_test, pred_prob)
RocCurveDisplay.from_estimator(pipe_svm, X_test, y_test)
auc(fpr,tpr)

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import auc

### Your code is here
# CalibrationDisplay.from_estimator(pipe_lr, X_test, y_test)

CalibrationDisplay.from_predictions(y_test,pipe_lr.predict_proba(X_test)[:,1],n_bins=15)
CalibrationDisplay.from_predictions(y_test, pred_prob, n_bins=15)

### Калибровочные кривые

Постройте кривые обеих моделей; для SVM используйте from_predictions с преобразованными оценками.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve


### Your code is here
CalibrationDisplay.from_estimator(pipe_lr, X_test, y_test,n_bins=25)



In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve


### Your code is here
CalibrationDisplay.from_predictions(y_test, pred_prob, n_bins=15)



Сравните кривые с диагональю. Откалибруйте именно SVM и постройте новую кривую.

In [ ]:
### Калибровка

from sklearn.calibration import CalibratedClassifierCV

### Your code is here
calibration = CalibratedClassifierCV(pipe_svm, method='isotonic', cv=5)
calibration.fit(X_train, y_train)

calibrated_probs = calibration.predict_proba(X_test)[:,1]


In [ ]:
### Новая кривая

### Your code is here

CalibrationDisplay.from_predictions(y_test,calibrated_probs,n_bins=25)
